In [2]:
from nbconvert.exporters import webpdf
import handcalcs.render


handcalcs.set_option("param_columns", 4)

In [7]:
%%tex param

#""" Inputs """

# gross area
A_g = 1 # mm2

# thickness of part
t = 10 # mm

# Yeald Stress
F_y = 210 # MPa
# Ultimate Yeald Stress
F_u = 300 # MPa

# Optins: Pin, Flange-connected Ts, Angles connected by one leg, stem-connected Ts, One bolt line coped Beams, Two bolt line coped beams, other
ConnectionType = "Pin"

#""" Constants """
phi_s = 0.9
phi_u = 0.75

\[
\begin{aligned}
A_{g} &= 1 \; \;\textrm{(mm2)}
 &t &= 10 \; \;\textrm{(mm)}
 &F_{y} &= 210 \; \;\textrm{(MPa)}
 &F_{u} &= 300 \; \;\textrm{(MPa)}
\\[10pt]
 \mathrm{ConnectionType} &= \mathrm{Pin} \; 
 &\phi_{s} &= 0.900 \; 
 &\phi_{u} &= 0.750 \;
\end{aligned}
\]


In [1]:
""" Not needed for pin conections """

def getEffectiveConectionMultiplyer(effectiveConection, ConnectionMethod = None, MemberType = None, WideFlange = None, NumTransversFastenersLines = None, WeldDirection = None, NumWelds = None, L = None, x_bar = None, w = 0):
    """
    :param effectiveConection: is the conection effective of distributed the tention force
    :param ConnectionMethod: conection Method. Options: Bolt, Weld
    :param MemberType: Options: WWF, W, M, S, L
    :param WideFlange: If WWF, W, M, or S shapes, is flange widths not less than two-thirds the depth
    :param NumTransversFastenersLines: Number of transvers lines of fasteners
    :param WeldDirection: Options:  Longitudinal, Transverse, Both
    :param NumWelds: number of Welds
    :param L: length of weld in direction of force
    :param x_bar: eccentricity of the weld with respect to centroid of the connected element
    :param w: numColumbs of element being welded at the welding location
    :return:
    """

    if effectiveConection:
        return 1
    if ConnectionMethod == "Bolt":
        if WideFlange and NumTransversFastenersLines >= 3 and (MemberType == "WWF" or MemberType == "W" or MemberType == "M" or MemberType == "S"):
            efectiveConectionMultiplyer = 0.9
        elif MemberType == "L":
            if NumTransversFastenersLines >= 4:
                efectiveConectionMultiplyer = 0.8
            else:
                efectiveConectionMultiplyer = 0.6
        else:
            if NumTransversFastenersLines >= 3:
                efectiveConectionMultiplyer = 0.85
            else:
                efectiveConectionMultiplyer = 0.75

    else:
        if WeldDirection == "Transverse":
            An = w * t

        elif WeldDirection == "Longitudinal":
            if L >= 2*w:
                An = w*t
            if 2*w > L >= w:
                An = 0.5 * w * t + 0.25 * L * t
            else:
                An = 0.75 * L * t

            if NumWelds == 1:
                if L >= w:
                    An += (1 - x_bar / L)*w*t
                else:
                    An += 0.5 * L * t

        else:
            An = w * t
            if L >= 2*w:
                An += w*t
            if 2*w > L >= w:
                An += 0.5 * w * t + 0.25 * L * t
            elif w > L:
                An += 0.75 * L * t


            if NumWelds == 1:
                if L >= w:
                    An += (1 - x_bar / L)*w*t
                else:
                    An += 0.5 * L * t

        efectiveConectionMultiplyer = An / A_n

    return efectiveConectionMultiplyer


"""Inputs"""
EffectiveConectionMultiplyer = getEffectiveConectionMultiplyer(True)

In [61]:
%%render
#""" only needed for Pin conections """

#""" Inputs """

# distance from the edge of the hole to the edge of the part normal to the tensile force
b_ToEadge = 30 # mm
# shortest distance parallel to the tensile force from the edge of the pinhole to the end of the tension member pin plate
a = 30 # mm
# diameter of pin
d = 10 # mm

#""" Calculations """

b_e = min(2 * t + 16, b_ToEadge) # mm
A_net = 2*t*b_e # mm2

A_nes = 2*t*(a + d/2) # mm2

<IPython.core.display.Latex object>

In [4]:
%%render
if ConnectionType == "Pin": T_r = min(phi_s * A_g * F_y, phi_u * A_net * F_u, 0.6 * phi_u * A_nes * F_u) # N

if ConnectionType == "Flange-connected Ts": U_t = 1.0
if ConnectionType == "Angles connected by one leg": U_t = 0.6
if ConnectionType == "stem-connected Ts": U_t = 0.6
if ConnectionType == "One bolt line coped Beams": U_t = 0.9
if ConnectionType == "Two bolt line coped Beams": U_t = 0.3
if ConnectionType == "Other": U_t = 1
if ConnectionType != "Pin": T_r = min(phi_s * A_g * F_y, phi_u * (U_t * A_n * F_u + 0.6 * A_gv * (F_y + F_u)/2), phi_u * A_ne * F_u) # N



NameError: name 'ConnectionType' is not defined